<div style="background-color: #1A5276; padding: 20px; border-radius: 10px; text-align: center; margin-bottom: 30px;">
    <h1 style="color: white; margin: 0;">Study &amp; Mastery Partner</h1>
    <h2 style="color: white; margin-top: 15px;">Study with AI — without outsourcing the thinking you're graded on</h2>
    <p style="color: white; margin-top: 10px; font-style: italic;">60–90 minutes · self-paced</p>
</div>

## What you walk out with

1. **A practice quiz** generated from YOUR own course material, with a hidden answer key you can self-test against.
2. **A Socratic study partner** that helps you get un-stuck *without* handing you the answer.
3. **An honest map of your understanding gaps**, from explaining a concept in your own words and having it checked.
4. **An integrity boundary + a concrete study plan** — bundled into one file you email to yourself.

## How this lab works

This lab has **two kinds of cells**:

- **Watch-along code cells** — just run them top to bottom; you don't edit anything.
- **🟢 EDIT ME cells** — you fill in values: your material, your stuck question, your study plan.

**The writing cells matter as much as the code cells.** A blank reflection cell means a missing plan. Don't skip them.

## What this lab is NOT

- **Not a way to get answers to graded work.** The whole point is to study *harder*, not to skip the work.
- **Not a coding tutorial.** There's some Python; you're not learning to code.
- **Not a replacement for reading and doing the work.** It's a tool to help you learn what you have to learn anyway.

Work top to bottom. Don't skip ahead.

---
# Part 0 — What this partner does (and doesn't) *(5 min)*

This partner has **three study modes**. Each one is built to help you *learn*, not to do the work for you:

| Mode | Use it when... | What it does |
|---|---|---|
| **Quiz Me** | It's the night before the exam and you want to test yourself | Generates a practice quiz from your material, with a hidden answer key |
| **Socratic Hint** | You're stuck on a problem at 11pm | Gives you the *next question to ask yourself* — never the answer |
| **Explain-Back** | You *think* you understand, but aren't sure | Checks your own explanation against the source and shows you the gaps |

Notice what's missing: there is no "write my essay" or "solve my problem set" mode. That's deliberate — see Part 6.

### 🟢 EDIT ME — Who you are and what you're studying

In [ ]:
student_name = "[Your name]"
course_label = "[Course code and name, e.g., BIO 210 Cell Biology]"

---
# Part 1 — Setup *(watch-along, ~8 min)*

Run the next two cells. You don't need to edit anything — these install the tools and connect to AWS once.

### 1.1 Install dependencies
*Takes ~30 seconds. No output is expected — it runs quietly.*

In [ ]:
%%capture
!pip install -q -r requirements.txt

### 1.2 Import tools and connect to Bedrock
*One handshake with AWS. After this, calling the AI is one line of code.*

In [ ]:
import boto3
import warnings
from datetime import datetime
from IPython.display import Markdown, display

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter

from mlu_utils.embeddings import NovaMultimodalEmbeddings
from mlu_utils.study_tools import build_modes

warnings.filterwarnings("ignore")

bedrock_runtime = boto3.client(service_name="bedrock-runtime", region_name="us-east-1")
embeddings = NovaMultimodalEmbeddings(client=bedrock_runtime)

# Safe defaults so the take-home export works even if you skip a mode.
my_quiz_answers = ""
my_gaps_to_review = ""
quiz = None

print("Ready.")

---
# Part 2 — Load YOUR material *(~8 min)*

This is the one variable you change to study **your** material.

The lab ships with a sample textbook chapter so it runs immediately. To study your own material instead: in the **file browser on the left**, open the `data/` folder, **drag your PDF in**, then change the filename below.

**Your file must be:** a text-based PDF (not a scan/photo of pages) and under ~50 pages.

### 🟢 EDIT ME — Point this at your material

In [ ]:
# Default = a sample textbook chapter so the lab runs out of the box.
# To use your own: drag a PDF into the data/ folder (left sidebar),
# then change this to its filename.
SOURCE_PDF = "data/persona2_cs_data_structures.pdf"
# e.g.  SOURCE_PDF = "data/my_lecture_notes.pdf"

### 2.2 Build your study index
*This reads your PDF, splits it into chunks, and indexes it so the AI can search it. Takes 30–90 seconds.*

In [ ]:
pages = PyPDFLoader(SOURCE_PDF).load()
print(f"Loaded {len(pages)} pages from {SOURCE_PDF}")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=60,
    separators=["\n\n", "\n", "(?<=\\. )", " ", ""],
    is_separator_regex=True,
)
chunks = splitter.split_documents(pages)
print(f"Split into {len(chunks)} chunks.")

print("Building searchable index... (30-90 seconds)")
vectordb = FAISS.from_documents(chunks, embeddings)
retriever = vectordb.as_retriever(search_kwargs={"k": 6})
print("Ready.")

> 💡 **Pause here.** Look at the page and chunk counts above. Do they match the document you meant to load? If you swapped in your own PDF, make sure it loaded the right number of pages before moving on.

---
# Part 3 — Mode 1: Quiz Me *(~15 min)*

Generates a practice quiz from your material — a mix of recall ("what is...") and conceptual ("why...", "what happens if...") questions. The answer key is **hidden** so you can test yourself first.

### 3.1 Build your three study modes
*One line — this wires up Quiz Me, Socratic Hint, and Explain-Back.*

In [ ]:
modes = build_modes(retriever, bedrock_runtime)
print("Three study modes ready: Quiz Me, Socratic Hint, Explain-Back.")

### 3.2 ▶ Generate your quiz
*Takes ~30 seconds. Only the questions are shown — the answer key stays hidden until you reveal it.*

In [ ]:
quiz = modes.quiz_me(num_questions=6)
display(Markdown("## Your Practice Quiz\n\n" + quiz.questions_md))

> 💡 **Pause here — this is the point.** Actually try the quiz *before* you reveal the answers. Don't scroll ahead. Write your answers in the cell below (or on paper). Testing yourself before you see the answer is what makes the studying stick.

### 🟢 EDIT ME — Your answers (before revealing the key)

In [ ]:
my_quiz_answers = """
1.
2.
3.
4.
5.
6.
"""

### 3.4 ▶ Reveal the answer key
*Run this only after you've attempted the quiz above.*

In [ ]:
display(Markdown("## Answer Key\n\n" + quiz.answer_key_md))

---
# Part 4 — Mode 2: Socratic Hint *(~15 min)*

When you're stuck, this partner will **not** give you the answer. It gives you the next question to ask yourself and points you to where in your material to look.

That's the point: getting the answer handed to you teaches you nothing. Working out the *next step* yourself is how you actually learn to solve it.

### 🟢 EDIT ME — What are you stuck on?

In [ ]:
stuck_question = """Paste the problem or question you're stuck on here.
Be specific — the more context you give, the better the hints."""

### 4.2 ▶ Get a Socratic hint
*Takes ~20 seconds.*

In [ ]:
hint = modes.socratic_hint(stuck_question)
display(Markdown(hint))

> 💡 **Pause here.** Read the guiding questions and try to answer *them* yourself before re-running with a follow-up. Go re-read the part of your material it pointed you to.
>
> If the partner ever just gives you the final answer, that's a known limitation of smaller AI models — not how it's supposed to behave. Notice it, and don't trust the answer blindly. That skepticism is itself worth learning.

---
# Part 5 — Mode 3: Explain-Back *(~15 min)*

The fastest way to find a hole in your understanding is to explain a concept in your own words and have it checked against the source.

The partner will point at your gaps and at the passage to re-read — it will **not** rewrite your explanation for you. Fixing the gap yourself is the part that makes it stick.

### 🟢 EDIT ME — Explain a concept in your own words

In [ ]:
concept_name = "[The concept you want to check, e.g., binary search]"

my_explanation = """Explain the concept in your own words here.
Don't look it up first — write what you actually think you know.
The gaps are the useful part."""

### 5.2 ▶ Check my understanding
*Takes ~20 seconds.*

In [ ]:
check = modes.explain_back(concept_name, my_explanation)
display(Markdown(check))

> 💡 **Pause here.** Read the gaps it found. Don't fix them by copying anything — go back to the passage it cited and re-derive the correction yourself. Then write those gaps down below so they end up in your study plan.

### 🟢 EDIT ME — The gaps you now need to study

In [ ]:
my_gaps_to_review = """[In your own words, the gaps or misconceptions you need to go back and study.]"""

---
# Part 6 — Where using AI would be cheating yourself *(15 min)*

**This is the most important section of the lab.**

There's a line between **studying with AI** and **letting AI do the thinking you're being graded on**. Everything in this lab so far has been on the right side of that line — quizzing yourself, getting hints, checking your understanding. But the same tool can be misused.

| ✅ Studying *with* AI (what this lab does) | ❌ Letting AI do the graded thinking (cheating yourself) |
|---|---|
| Quiz yourself on the material | Paste the take-home exam question and submit the output |
| Ask for a hint when stuck | Have it solve the problem set you'll be graded on |
| Check your own explanation for gaps | Have it write the essay and turn it in as yours |
| Get pointers on what to re-read | Skip the reading and have it summarize so you never engage |

The difference isn't the tool — it's whether *you* still do the thinking the assignment is meant to build.

Crossing that line doesn't just risk a policy violation. It cheats *you* out of the learning you're paying for and will need later.

### 🟢 EDIT ME — Your integrity boundary

In [ ]:
# 1. The line: where, in THIS course, would using AI cross from studying
#    into doing the thinking you're being graded on?
my_integrity_line = "[e.g., 'Using AI to write my argument essay — the argument IS the grade.']"

# 2. Your course's actual policy: what does YOUR syllabus / instructor say
#    about AI use? (If you don't know, that's the action item — go find out.)
my_course_ai_policy = "[Quote or paraphrase your course's AI policy. Write 'I need to check' if unsure.]"

# 3. The honest self-check: one assignment in this course where you will
#    deliberately NOT use AI, because the struggle is the learning.
my_protected_work = "[The assignment you'll do without AI, and why the struggle matters to YOU.]"

---
# Part 7 — Your study plan + take-home artifact *(10 min)*

Turn what you just did into a concrete plan. Be specific: not "study more," but *when*, *what*, and *how*.

### 🟢 EDIT ME — Your study plan

In [ ]:
study_topic   = "[The topic or exam you're studying for]"
study_when    = "[YYYY-MM-DD and time you'll sit down to study — be specific]"
study_how     = "[Which mode(s) you'll use and how — e.g., 'Quiz Me on Ch.4, then Explain-Back the 2 gaps it finds']"
my_commitment = "[One sentence: a study action you commit to, that uses AI to learn — not to shortcut.]"

### 7.2 Save your study plan as a take-home artifact
*This bundles your quiz, your gaps, your integrity boundary, and your plan into one markdown file you can download.*

In [ ]:
filename = f"my_study_plan_{datetime.now().strftime('%Y%m%d_%H%M')}.md"

with open(filename, "w") as f:
    f.write("# My Study & Mastery Plan\n\n")
    f.write(f"**Student:** {student_name}\n\n")
    f.write(f"**Course:** {course_label}\n\n")
    f.write(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M')}\n\n")
    f.write(f"**Source material:** `{SOURCE_PDF}`\n\n")

    f.write("---\n\n## My Commitment\n\n")
    f.write(f"On **{study_when}**, I will study **{study_topic}**:\n\n> {my_commitment}\n\n")

    f.write("---\n\n## My Integrity Boundary\n\n")
    f.write(f"- **My line:** {my_integrity_line}\n")
    f.write(f"- **Course AI policy:** {my_course_ai_policy}\n")
    f.write(f"- **Work I'll protect (no AI):** {my_protected_work}\n\n")

    f.write("---\n\n## My Study Plan\n\n")
    f.write(f"- **Topic:** {study_topic}\n")
    f.write(f"- **When:** {study_when}\n")
    f.write(f"- **How:** {study_how}\n\n")

    if quiz is not None:
        f.write("---\n\n## My Practice Quiz\n\n")
        f.write(quiz.questions_md + "\n\n### Answer Key\n\n" + quiz.answer_key_md + "\n\n")
        f.write("### My Attempt\n\n" + my_quiz_answers + "\n\n")

    f.write("---\n\n## Gaps To Review\n\n")
    f.write(my_gaps_to_review + "\n")

print(f"Saved to: {filename}")
print("Right-click the file in the left browser → Download.")
print("Then email it to yourself with subject: 'My study plan'.")

---
## What you walk out with

✅ A practice quiz built from your own material, with an answer key
✅ A Socratic partner that helped you get un-stuck without doing it for you
✅ An honest list of your understanding gaps
✅ A clear line for where using AI would cross into cheating yourself
✅ A specific, dated study plan
✅ A downloadable markdown artifact bundling all of the above

## What to do next

1. **Email yourself the artifact.** Calendar reminders get ignored; a study plan in your inbox doesn't.
2. **Actually sit down on the date you committed to** and run the plan — Quiz Me, then Explain-Back the gaps.
3. **Re-run this notebook with your next topic.** The more of your own material you bring, the more useful it gets.